# Extraire les objets OSM de la région

Nous extrayons les catégories utiles : équipements/services, stationnements vélo, parkings, sites touristiques et sites de recharge.

L'extraction utilise les deux PBF Geofabrik déjà téléchargés. Elle conserve des **points, lignes et surfaces**, y compris les multipolygones OSM. Les objets sont sélectionnés s'ils touchent la région ou sa marge de 1 km.

Choisir **Python (IGN / OSM)** après le notebook 01. La première exécution parcourt deux fois chaque fichier pour assembler les surfaces et peut prendre plusieurs minutes. Les prochains lancements réutilisent l'export si le manifeste correspond aux mêmes sources et règles.

[Documentation PyOsmium sur les surfaces](https://docs.osmcode.org/pyosmium/latest/user_manual/03-Working-with-Geometries/).

In [1]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
from IPython.display import display

current_directory = Path.cwd().resolve()
project_directory = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "regional-data" / "AGENTS.md").exists():
        project_directory = directory
        break
if project_directory is None:
    raise FileNotFoundError("Open this notebook from the repository directory.")
module_directory = project_directory / "regional-data"
output_directory = module_directory / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)

import osmium
from shapely import wkb

region = gpd.read_file(output_directory / "aura_region.geojson").geometry.iloc[0]
counting_area = gpd.read_file(output_directory / "aura_counting_area.geojson").geometry.iloc[0]
input_paths = [
    project_directory / "ign-osm" / "data" / "raw" / "osm" / "auvergne-261016.osm.pbf",
    project_directory / "ign-osm" / "data" / "raw" / "osm" / "rhone-alpes-261016.osm.pbf",
]
objects_path = output_directory / "osm_objects.gpkg"
errors_path = output_directory / "osm_extraction_errors.csv"
manifest_path = output_directory / "osm_extraction_manifest.json"

## 1. Définir les catégories

Les listes ci-dessous définissent les variables : « équipements » ne signifie pas tous les tags `amenity` possibles. Les parkings, parkings vélo et recharges sont séparés du comptage d'équipements général.

Les sites touristiques comprennent les attractions, musées, galeries, points de vue, zoos, aquariums, parcs à thème et œuvres d'art. Les hôtels et panneaux d'information ne sont pas comptés dans cette définition.

Un objet peut appartenir à deux catégories, mais sera compté au plus une fois **par catégorie**.

In [2]:
amenity_values = {
    "restaurant", "cafe", "bar", "fast_food", "pub", "food_court",
    "school", "kindergarten", "college", "university", "library",
    "community_centre", "arts_centre", "theatre", "cinema", "hospital",
    "clinic", "doctors", "dentist", "pharmacy", "bank", "atm",
    "post_office", "marketplace", "townhall", "police", "fire_station",
    "social_facility", "place_of_worship",
}
tourism_values = {"attraction", "museum", "gallery", "viewpoint", "zoo", "aquarium", "theme_park", "artwork"}


def object_categories(tags):
    categories = []
    amenity = tags.get("amenity")
    if amenity in amenity_values:
        categories.append("amenity")
    if amenity == "bicycle_parking":
        categories.append("bike_parking")
    if amenity == "parking":
        categories.append("parking")
    if amenity == "charging_station":
        categories.append("charging_site")
    if tags.get("tourism") in tourism_values:
        categories.append("tourism")
    return categories

rules = {"version": 1, "amenities": sorted(amenity_values), "tourism": sorted(tourism_values)}
expected_manifest = {
    "rules": rules,
    "source_files": [{"name": path.name, "size_bytes": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns} for path in input_paths],
    "counting_area_wkb": counting_area.wkb_hex,
}
reuse_export = False
if objects_path.exists() and manifest_path.exists():
    previous_manifest = json.loads(manifest_path.read_text())
    reuse_export = previous_manifest.get("inputs") == expected_manifest
print(f"Reuse existing extraction: {reuse_export}")

Reuse existing extraction: True


## 2. Lire les objets et leurs formes

Un point OSM est pris directement. Une voie ouverte est convertie en ligne. Les voies fermées et relations multipolygones sont traitées comme surfaces, sans compter aussi leur voie d'origine.

Les erreurs de géométrie sont consignées, jamais transformées en objets fictifs. Un compteur vérifie aussi les surfaces attendues mais non reconstruites. Les identifiants comportent le type (`node`, `way`, `relation`) : les nombres seuls ne sont pas uniques entre types.

Les fichiers peuvent se chevaucher. Nous regrouperons les objets ayant le même type et identifiant OSM, en privilégiant la version OSM la plus élevée. Cela évite les doubles comptes entre extraits, sans prétendre identifier tous les doublons entre objets OSM différents.

In [3]:
geometry_factory = osmium.geom.WKBFactory()


def read_extract(path):
    records = []
    errors = []
    expected_areas = set()
    found_areas = set()
    processor = (
        osmium.FileProcessor(str(path)).with_locations().with_areas()
        .with_filter(osmium.filter.KeyFilter("amenity", "tourism"))
    )
    for item in processor:
        categories = object_categories(item.tags)
        if not categories:
            continue
        if item.is_area():
            object_type = "way" if item.from_way() else "relation"
            object_id = item.orig_id()
            found_areas.add(f"{object_type}/{object_id}")
        elif item.is_node():
            object_type, object_id = "node", item.id
        elif item.is_way():
            if item.is_closed():
                expected_areas.add(f"way/{item.id}")
                continue
            object_type, object_id = "way", item.id
        elif item.is_relation():
            expected_areas.add(f"relation/{item.id}")
            continue
        else:
            continue
        object_uid = f"{object_type}/{object_id}"
        try:
            if item.is_node():
                geometry = wkb.loads(geometry_factory.create_point(item.location), hex=True)
            elif item.is_area():
                geometry = wkb.loads(geometry_factory.create_multipolygon(item), hex=True)
            else:
                geometry = wkb.loads(geometry_factory.create_linestring(item), hex=True)
            if geometry.is_empty or not geometry.is_valid:
                raise ValueError("Empty or invalid geometry")
        except (RuntimeError, ValueError, osmium.InvalidLocationError) as error:
            errors.append({"source_file": path.name, "osm_uid": object_uid, "reason": str(error)})
            continue
        if not geometry.intersects(counting_area):
            continue
        records.append({
            "osm_uid": object_uid, "osm_type": object_type, "osm_id": str(object_id),
            "osm_version": item.version, "name": item.tags.get("name", ""),
            "amenity": item.tags.get("amenity", ""), "tourism": item.tags.get("tourism", ""),
            "categories": ",".join(categories), "access": item.tags.get("access", ""),
            "capacity_raw": item.tags.get("capacity", ""),
            "operator": item.tags.get("operator", ""),
            "source_files": path.name, "inside_region": geometry.intersects(region),
            "geometry": geometry,
        })
    for missing_uid in sorted(expected_areas - found_areas):
        errors.append({"source_file": path.name, "osm_uid": missing_uid, "reason": "Tagged area or relation not reconstructed"})
    return records, errors

if not reuse_export:
    all_records, all_errors = [], []
    for path in input_paths:
        print(f"Reading {path.name}...", flush=True)
        records, errors = read_extract(path)
        all_records.extend(records)
        all_errors.extend(errors)
        print(f"Selected {len(records)} objects; logged {len(errors)} geometry issues", flush=True)
    objects = gpd.GeoDataFrame(all_records, geometry="geometry", crs="EPSG:4326")
    source_files = objects.groupby("osm_uid")["source_files"].agg(lambda values: ",".join(sorted(set(values))))
    duplicate_extract_rows = len(objects) - objects["osm_uid"].nunique()
    objects = objects.sort_values(["osm_uid", "osm_version"]).drop_duplicates("osm_uid", keep="last").copy()
    objects["source_files"] = objects["osm_uid"].map(source_files)
    errors = pd.DataFrame(all_errors, columns=["source_file", "osm_uid", "reason"])
else:
    objects = gpd.read_file(objects_path)
    errors = pd.read_csv(errors_path)
    duplicate_extract_rows = previous_manifest["duplicate_extract_rows"]
print(f"Unique OSM objects: {len(objects)}")

Unique OSM objects: 192803


## 3. Identifier certains doublons point/surface

Un lieu peut être cartographié à la fois par un point et une surface. Nous signalons comme doublon de comptage un point **inclus dans une surface avec les mêmes catégories, le même nom non vide et les mêmes tags amenity/tourism**. Le point reste livré, avec `countable=False` et l'identifiant de la surface dans `duplicate_of`.

Ce rapprochement est volontairement limité : des lieux anonymes ou décrits différemment peuvent encore être des doublons physiques. Les décomptes restent ceux des objets OSM recensés selon nos règles, pas un inventaire exhaustif du terrain.

In [4]:
if not reuse_export:
    objects["countable"] = True
    objects["duplicate_of"] = ""
    metric_objects = objects.to_crs(epsg=2154)
    points = metric_objects.loc[metric_objects.geometry.geom_type == "Point"]
    areas = metric_objects.loc[metric_objects.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    for index, point in points.iterrows():
        if not point["name"].strip():
            continue
        candidates = areas.sindex.query(point.geometry, predicate="intersects")
        for position in candidates:
            area = areas.iloc[position]
            same_name = point["name"].strip().casefold() == area["name"].strip().casefold()
            same_tags = all(point[key] == area[key] for key in ["categories", "amenity", "tourism"])
            if same_name and same_tags:
                objects.loc[index, "countable"] = False
                objects.loc[index, "duplicate_of"] = area["osm_uid"]
                break
    objects["capacity"] = pd.to_numeric(objects["capacity_raw"].where(objects["capacity_raw"].str.fullmatch(r"[0-9]+")), errors="coerce").astype("Int64")
    objects.to_file(objects_path, layer="objects", driver="GPKG")
    errors.to_csv(errors_path, index=False)
    manifest = {"inputs": expected_manifest, "object_count": len(objects), "geometry_issue_count": len(errors), "duplicate_extract_rows": duplicate_extract_rows, "point_area_duplicates": int((~objects["countable"]).sum())}
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

assert objects["osm_uid"].is_unique
assert objects.geometry.is_valid.all()
assert not objects.geometry.is_empty.any()
assert objects.geometry.intersects(counting_area).all()
saved_objects = gpd.read_file(objects_path)
assert set(saved_objects["osm_uid"]) == set(objects["osm_uid"])
print(f"Recorded geometry issues: {len(errors)}")
print(f"Overlap rows removed: {duplicate_extract_rows}")
print(f"Point/area duplicates excluded from counts: {(~objects['countable']).sum()}")
display(objects.drop(columns="geometry").head())
display(objects["categories"].str.get_dummies(sep=",").sum().rename("mapped_object_count").to_frame())

Recorded geometry issues: 35
Overlap rows removed: 64
Point/area duplicates excluded from counts: 70


,osm_uid,osm_type,osm_id,osm_version,name,amenity,tourism,categories,access,capacity_raw,operator,source_files,inside_region,countable,duplicate_of,capacity
0,node/10000199953,node,10000199953,5,,bicycle_parking,,bike_parking,,12,,rhone-alpes-261016.osm.pbf,True,True,,12.0
1,node/10000199956,node,10000199956,2,,bicycle_parking,,bike_parking,,6,,rhone-alpes-261016.osm.pbf,True,True,,6.0
2,node/10000199958,node,10000199958,2,,bicycle_parking,,bike_parking,,10,,rhone-alpes-261016.osm.pbf,True,True,,10.0
3,node/10000199959,node,10000199959,2,,bicycle_parking,,bike_parking,,12,,rhone-alpes-261016.osm.pbf,True,True,,12.0
4,node/10000199961,node,10000199961,2,,bicycle_parking,,bike_parking,,10,,rhone-alpes-261016.osm.pbf,True,True,,10.0


,mapped_object_count
amenity,67044
bike_parking,24124
charging_site,2969
parking,87338
tourism,11379


## 4. Livrer les catégories comme couches séparées

Un seul GeoPackage peut contenir plusieurs couches, comme plusieurs tables dans une base de données. Le fichier `osm_layers.gpkg` permet de choisir directement les équipements, parkings vélo, parkings, sites de recharge ou sites touristiques sans refaire un filtre.

Les objets non comptables restent présents pour l'audit ; filtrer `countable=True` pour reproduire les indicateurs. Un objet appartenant à plusieurs catégories apparaît dans chacune des couches concernées.


In [5]:
layer_names = {"amenity": "amenities", "bike_parking": "bike_parking", "parking": "parking", "charging_site": "charging_sites", "tourism": "tourism_sites"}
for category, layer_name in layer_names.items():
    category_rows = objects["categories"].str.split(",").apply(lambda values: category in values)
    objects.loc[category_rows].to_file(
        output_directory / "osm_layers.gpkg", layer=layer_name, driver="GPKG"
    )
    saved_layer = gpd.read_file(output_directory / "osm_layers.gpkg", layer=layer_name)
    assert set(saved_layer["osm_uid"]) == set(objects.loc[category_rows, "osm_uid"])
print("Five thematic layers saved and verified.")


Five thematic layers saved and verified.


## Limites

La marge permet de conserver des équipements extérieurs proches d'une station régionale, mais la couverture réelle dépend des contours des extraits Geofabrik. Le notebook 03 signalera les rayons dont le carré englobant dépasse le contour IGN ; la couverture exacte par les extraits historiques reste à vérifier pour ces stations.

Les erreurs recensées peuvent concerner des objets hors de la région, car elles surviennent avant de pouvoir les localiser. Elles seront signalées dans les indicateurs plutôt qu'ignorées. Les catégories comptent les objets recensés ; les capacités inconnues restent inconnues.

Données : © OpenStreetMap contributors, licence ODbL 1.0, extraits Geofabrik. Dates et empreintes des PBF : `ign-osm/sources.csv`. Les objets de recharge sont des **sites OSM**, pas automatiquement des bornes ou points de recharge.